# Ask Coresight: search model v2 (about 15-25 minutes)

v1 taught us two things: the same question asked by several studies in other words must be **one class** (v1 learned to
push them apart and became confidently wrong), and the questions analysts ask most must **count more**. v2 trains two
candidates with both fixes, shrinks each for the server and downloads **bge-cip-v2.zip**; the choice is made in Ask's
full pipeline afterwards.

1. Runtime → Change runtime type → **L4** (or A100) GPU → Save.
2. Optional, for better training: Colab Secrets (key icon, left) → `GEMINI_API_KEY` = your AI Studio key, notebook access on.
3. Runtime → **Run all**; at **Choose Files** pick `catalogue.json` (the new one, with question groups).
4. **Keep the laptop open** until the zip downloads; the GPU is released 15 minutes after.

In [ ]:
# quiet: Colab's own packages (gradio, diffusers) are not used here, so pip's notes about them are hidden
!pip -q install "sentence-transformers==3.3.1" onnx onnxscript onnxruntime "google-genai>=1.0" > /dev/null 2>&1   # 3.3.1: the version that trained here on 12 Nov
import torch, json, random, time, os, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: Runtime → Change runtime type → GPU")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from google.colab import files
if not os.path.exists("catalogue.json"):
    files.upload()   # choose catalogue.json
data = json.load(open("catalogue.json"))
docs, golden = data["docs"], data["golden"]
TRAIN_ASKS = 6   # asks 1-6 train and index; asks 7-8 are the unseen test
print(len(docs), "survey questions,", len(golden), "golden questions")

In [ ]:
from sentence_transformers import SentenceTransformer

def index_texts():
    owners, texts = [], []
    for n, d in enumerate(docs):
        for t in [d["embed"]] + d["asks"][:TRAIN_ASKS] + d.get("extra", []):
            owners.append(n); texts.append(t)
    return np.array(owners), texts

def score(encode, label):
    """Share right on unseen asks and golden, and at each confidence gap (best vs next *different* survey question)."""
    owners, texts = index_texts()
    vec = encode(texts)
    out = {}
    tests = {"unseen": ([a for d in docs for a in d["asks"][TRAIN_ASKS:]], [n for n, d in enumerate(docs) for _ in d["asks"][TRAIN_ASKS:]]),
             "golden": ([g["q"] for g in golden], golden)}
    for name, (queries, wants) in tests.items():
        sims = encode(queries) @ vec.T
        rows = []
        for row, want in zip(sims, wants):
            per = np.full(len(docs), -1.0); np.maximum.at(per, owners, row)
            order = np.argsort(-per)[:50]; top = order[0]
            other = next((o for o in order[1:] if docs[o]["group"] != docs[top]["group"]), order[1])   # the next *different* question
            right = docs[top]["group"] == docs[want]["group"] if name == "unseen" else any(w in docs[top]["text"].lower() for w in want["want"])
            rows.append((per[top] - per[other], right))
        rows = np.array(rows, float)
        curve = {g: (float(rows[rows[:, 0] >= g, 1].mean()) if (rows[:, 0] >= g).any() else 0.0, float((rows[:, 0] >= g).mean()))
                 for g in (0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.15)}
        out[name] = curve
        print(f"{label:34} {name:7} right {curve[0.0][0]:.1%} · " + " · ".join(f"gap≥{g}: {r:.1%} on {c:.0%}" for g, (r, c) in list(curve.items())[1:]))
    return out

def st_encoder(model):
    return lambda texts: model.encode(texts, batch_size=256, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

results = {}   # today's model was measured in the first run: unseen 39.0%, golden 58.3%

In [ ]:
# Optional: Gemini writes 4 more example questions per survey question (training only, never the test)
from google.colab import userdata
try:
    KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    KEY = None
if KEY:
    from google import genai
    client = genai.Client(api_key=KEY)
    MODEL = "gemini-2.5-flash"   # change if AI Studio lists a newer Flash model
    PROMPT = ("For each numbered consumer-survey question, write 4 short, different questions an analyst, a retail client or a "
              "colleague would type to get its figures: casual, precise, a decision, about one answer or a group. Never copy "
              "the survey's wording. Reply as JSON: {\"rows\": [{\"i\": number, \"asks\": [4 strings]}]}.")
    for start in range(0, len(docs), 25):
        part = docs[start:start + 25]
        listing = "\n".join(f"{i}. {d['text'][:200]}" for i, d in enumerate(part))
        for attempt in range(5):
            try:
                reply = client.models.generate_content(model=MODEL, contents=PROMPT + "\n\n" + listing,
                                                       config={"response_mime_type": "application/json"})
                for row in json.loads(reply.text)["rows"]:
                    if 0 <= row["i"] < len(part):
                        part[row["i"]]["extra"] = [a.strip()[:150] for a in row["asks"][:4]]
                break
            except Exception as exc:
                print("retrying:", str(exc)[:120]); time.sleep(20 * (attempt + 1))
        print(f"{min(start + 25, len(docs))} of {len(docs)}", end=" · ")
    json.dump({"docs": docs, "golden": golden}, open("catalogue_extra.json", "w"))
    print("\nextra asks for", sum(bool(d.get("extra")) for d in docs), "survey questions")
else:
    print("No GEMINI_API_KEY in Colab Secrets: training on the catalogue's own asks only")

In [ ]:
# v2: near-twins (the same question in several studies) are one class, never each other's wrong answer; questions
# analysts ask for longer (more waves) get more examples; look-alikes only from genuinely different questions
base = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
repeat = lambda d: 1 + (d["waves"] >= 10) + (d["waves"] >= 50)   # weekly tracker questions count up to 3x
anchors = [(t, n) for n, d in enumerate(docs) for t in d["asks"][:TRAIN_ASKS] + [d["name"]] + d["aliases"] + d.get("extra", []) if t
           for _ in range(repeat(d))]
owners, texts = index_texts()
vec = st_encoder(base)(texts)
sims = st_encoder(base)([a for a, _ in anchors]) @ vec.T
hard = []
for row, (_, n) in zip(sims, anchors):
    per = np.full(len(docs), -1.0); np.maximum.at(per, owners, row)
    hard.append(next((int(o) for o in np.argsort(-per) if docs[o]["group"] != docs[n]["group"]), (n + 1) % len(docs)))
print(len(anchors), "training pairs over", len({d["group"] for d in docs}), "question groups")

def train(epochs, with_hard, lr=3e-5, batch=128, scale=20.0):
    """In-batch contrastive training: each typed question nearer its own survey question than every other question in
    the batch (never a near-twin: one per group a batch), and than its hard look-alike when with_hard."""
    model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
    items = list(zip(anchors, hard))
    steps = []
    for e in range(epochs):
        random.Random(e).shuffle(items)
        current, groups = [], set()
        for (t, n), h in items:
            g = docs[n]["group"]
            if g in groups or (with_hard and docs[h]["group"] in groups): continue
            current.append((t, n, h)); groups.add(g)
            if len(current) == batch:
                steps.append(current); current, groups = [], set()
        if len(current) > 1:
            steps.append(current)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    warm = max(1, len(steps) // 10)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, 1 - s / len(steps)))
    model.train()
    emb = lambda ts: torch.nn.functional.normalize(model({k: v.to(DEVICE) for k, v in model.tokenize(ts).items() if torch.is_tensor(v)})["sentence_embedding"], dim=-1)
    started = time.time()
    for step in steps:
        a = emb([t for t, _, _ in step])
        targets = [docs[n]["embed"] for _, n, _ in step] + ([docs[h]["embed"] for _, _, h in step] if with_hard else [])
        loss = torch.nn.functional.cross_entropy(a @ emb(targets).T * scale, torch.arange(len(step), device=DEVICE))
        loss.backward(); opt.step(); sched.step(); opt.zero_grad()
    print(f"trained {epochs} epoch(s){' + look-alikes' if with_hard else ''} in {time.time() - started:.0f}s")
    model.eval()
    return model

models = {}
for epochs, with_hard in [(3, False), (3, True)]:
    label = f"v2 {epochs} ep{' + look-alikes' if with_hard else ''}"
    models[label] = train(epochs, with_hard)
    results[label] = score(st_encoder(models[label]), label)

In [ ]:
# Every candidate shrunk to the server's format (8-bit ONNX, first-token vector) and checked with the server's code path;
# the choice between them is made in Ask's full pipeline, not here (search-alone scores misled in v1)
import onnxruntime, tokenizers, shutil
from onnxruntime.quantization import quantize_dynamic, QuantType
os.makedirs("bge-cip-v2", exist_ok=True)
for label, model in models.items():
    folder = "bge-cip-v2/" + label.replace(" ", "-").replace("+", "plus")
    os.makedirs(folder, exist_ok=True)
    bert = model[0].auto_model.cpu().eval()
    dummy = model.tokenize(["a question"])
    torch.onnx.export(bert, (dummy["input_ids"], dummy["attention_mask"], dummy["token_type_ids"]), "full.onnx",
                      input_names=["input_ids", "attention_mask", "token_type_ids"], output_names=["last_hidden_state"],
                      dynamic_axes={k: {0: "batch", 1: "tokens"} for k in ["input_ids", "attention_mask", "token_type_ids", "last_hidden_state"]},
                      opset_version=17, dynamo=False)
    quantize_dynamic("full.onnx", folder + "/model_optimized.onnx", weight_type=QuantType.QInt8)
    model.tokenizer.save_pretrained("tok"); shutil.copy("tok/tokenizer.json", folder + "/tokenizer.json")
    words = tokenizers.Tokenizer.from_file(folder + "/tokenizer.json")
    words.enable_truncation(512); words.enable_padding(pad_id=0, pad_token="[PAD]")
    options = onnxruntime.SessionOptions(); options.intra_op_num_threads = options.inter_op_num_threads = 1
    session = onnxruntime.InferenceSession(folder + "/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
    def onnx_encode(texts):   # api/ask_model.py embed, line for line
        out = np.zeros((len(texts), 384), np.float32)
        order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
        for s in range(0, len(order), 32):
            b = order[s:s + 32]; coded = words.encode_batch([texts[i] for i in b])
            ids = np.array([c.ids for c in coded], np.int64)
            out[b] = session.run(None, {"input_ids": ids, "attention_mask": np.array([c.attention_mask for c in coded], np.int64),
                                        "token_type_ids": np.zeros_like(ids)})[0][:, 0]
        return out / np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-9)
    results[label + " · 8-bit"] = score(onnx_encode, label + " · 8-bit")
json.dump({k: {t: {str(g): v for g, v in c.items()} for t, c in r.items()} for k, r in results.items()}, open("bge-cip-v2/results.json", "w"), indent=1)
if os.path.exists("catalogue_extra.json"): shutil.copy("catalogue_extra.json", "bge-cip-v2/")
shutil.make_archive("bge-cip-v2", "zip", "bge-cip-v2")
print("READY: bge-cip-v2.zip is downloading to your Downloads folder.")
print("If the download does not start: left bar → folder icon → content → bge-cip-v2.zip → ⋮ → Download (within 15 minutes).")
files.download("bge-cip-v2.zip")

In [ ]:
# Stop paying once the download has had plenty of time: an idle GPU still uses units (15 min on an L4 ≈ 0.5)
import time
from google.colab import runtime
print("Releasing the GPU in 15 minutes. If bge-cip-v2.zip is not in Downloads yet: folder icon → content → bge-cip-v2.zip → ⋮ → Download")
time.sleep(900)
runtime.unassign()